# Integrate Valuations with Matched Players

This notebook merges player valuation data with the matched players table using `player_id` and season dates.

## 1. Import Dependencies

In [21]:
import pandas as pd

## 2. Read the Datasets

In [26]:
valuations = pd.read_csv('../datasets/Football Data from Transfermarkt/player_valuations.csv')
matched_players = pd.read_csv('matched_players_full.csv')
print(f"Valuations records: {len(valuations)}")
print(f"Matched players records: {len(matched_players)}")

Valuations records: 435443
Matched players records: 1012


## 3. Parse and Extract Season Dates

Handle both traditional mid-to-mid year seasons and calendar year seasons

In [27]:
valuations['date'] = pd.to_datetime(valuations['date'])

def get_season_start_end(season_str):
    # Handles formats like '2022/2023', '2024', etc.
    if '/' in str(season_str):
        start, end = season_str.split('/')
        start_year = int(start)
        end_year = int(end)
        return pd.Timestamp(f'{start_year}-07-01'), pd.Timestamp(f'{end_year}-06-30')
    elif str(season_str).isdigit():
        year = int(season_str)
        return pd.Timestamp(f'{year}-01-01'), pd.Timestamp(f'{year}-12-31')
    else:
        return pd.NaT, pd.NaT

matched_players[['season_start', 'season_end']] = matched_players['Seasons'].apply(
    lambda x: pd.Series(get_season_start_end(x))
)


## 4. Merge and filter valuations within season

In [28]:
merged = pd.merge(
    matched_players,
    valuations,
    on='player_id',
    how='left',
    suffixes=('', '_valuation')
)

in_season = merged[
    (merged['date'] >= merged['season_start']) &
    (merged['date'] <= merged['season_end'])
].copy()

print(f"Total matched records: {len(merged)}")
print(f"Records with valuations in season: {len(in_season)}")


Total matched records: 16091
Records with valuations in season: 1230


## 5. For each player and season, keep the latest valuation

In [34]:
in_season.sort_values(['player_id', 'season_start', 'date'], inplace=True)
latest_valuation = in_season.groupby(['player_id', 'season_start', 'season_end'], as_index=False).last()
print(f"Total records after selecting latest valuation: {len(latest_valuation)}")


Total records after selecting latest valuation: 705


## 6. Save the integrated table

In [35]:
# Select only the required columns
columns_to_save = ['Players', 'player_id', 'Seasons', 'season_start', 'season_end', 'Matches', 'Goals', 'Assists', 'Seasons Ratings', 'market_value_in_eur', 'date', 'Teams']
latest_valuation = latest_valuation[columns_to_save]

latest_valuation.to_csv('matched_players_with_valuations.csv', index=False)
print("Integrated table saved as 'matched_players_with_valuations.csv'")

Integrated table saved as 'matched_players_with_valuations.csv'


## 7. Show a sample

In [36]:
latest_valuation.head()

,Players,player_id,Seasons,season_start,season_end,Matches,Goals,Assists,Seasons Ratings,market_value_in_eur,date,Teams
0,Rodri,8163,2016/2017,2016-07-01,2017-06-30,31,1,2,7.1,250000.0,2017-06-05,Villarreal
1,Cristiano Ronaldo,8198,2015/2016,2015-07-01,2016-06-30,48,51,15,8.6,15000000.0,2016-06-01,Real Madrid
2,Cristiano Ronaldo,8198,2016/2017,2016-07-01,2017-06-30,46,42,11,7.9,15000000.0,2017-06-05,Real Madrid
3,Cristiano Ronaldo,8198,2017/2018,2017-07-01,2018-06-30,43,28,10,7.9,15000000.0,2018-05-30,Juventus
4,Cristiano Ronaldo,8198,2018/2019,2018-07-01,2019-06-30,44,36,5,7.9,15000000.0,2019-06-06,Juventus
